In [18]:
%cd E:\SRP\SRP-2025-Project
import os,sys
notebook_dir = os.getcwd()
path = os.path.abspath(os.path.join(notebook_dir, "Code"))
sys.path.append(path)
import optuna
import torch
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score,precision_recall_curve
from generateSplits import generateSplits
from Dataset import ModelDataset
from model import Model
from torch.utils.data import DataLoader
from torch import nn
from copy import deepcopy
from trainModel import evaluate_roc_auc

E:\SRP\SRP-2025-Project


In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

metadata = pd.read_csv("../Datasets/BreastDCEDL_spy1/BreastDCEDL_spy1_metadata.csv")
train_df,val_df = generateSplits(metadata,0.2,seed=42)
train_df = train_df[["pid","pCR","ER","PR","HER2"]].set_index("pid",drop=True)
val_df = val_df[["pid","pCR","ER","PR","HER2"]].set_index("pid",drop=True)
skf = StratifiedKFold(n_splits=4,shuffle=True,random_state=42)

In [3]:
best_params = {'lr': 0.010203264127576701, 'weight_decay': 0.004617603477095672, 'batch_size': 8, 'optimiser_name': 'Adam'}

In [4]:
def evaluate_average_precision(model,val_loader):
    model.eval()
    y_true=[]
    y_score=[]
    
    with torch.no_grad():
        for images,mols,labels in val_loader:
            images=images.to(device)
            mols = mols.to(device)
            labels=labels.to(device)
            logits = model(images,mols)
            score = torch.nn.functional.softmax(logits,dim=1)[:,1]
            y_true.extend(labels.cpu().numpy())
            y_score.extend(score.cpu().numpy())
    return average_precision_score(y_true,y_score)

In [13]:
def trainModel(model:nn.Module, train_loader,optimiser=None,device=torch.device("cpu"),num_epochs=10,val_loader=None,patience=5,scheduler=None):
    model = model.to(device)
    loss_fn = torch.nn.CrossEntropyLoss()
    best_average_precision = 0
    stale_epochs = 0
    best_model_state_dict = None
    if optimiser is None:
        optimiser = torch.optim.AdamW(model.parameters(),lr=1e-3)
    
    for epoch in range(num_epochs):
        model.train()
        losses = torch.tensor(0.0, device=device)
        for images, mol, labels in train_loader:
            images = images.to(device)
            mol = mol.to(device)
            labels = labels.to(device)
            optimiser.zero_grad()
            logits = model(images,mol)
            loss:torch.Tensor = loss_fn(logits,labels)
            loss.backward()
            optimiser.step()
            losses+=loss.detach()
        avg_loss = (losses/len(train_loader)).item()
        print(f"Epoch {epoch} Done. Avg Loss: {avg_loss:.4f}")
        
        if val_loader is not None:
            average_precision = evaluate_average_precision(model,val_loader)
            print(f"Average Precision: {average_precision}")
            
            if average_precision > best_average_precision:
                best_average_precision = average_precision
                stale_epochs=0
                best_model_state_dict=deepcopy(model.state_dict())
            else:
                stale_epochs+=1
                if stale_epochs>=patience:
                    print(f"Early stopping triggered at epoch {epoch} (best AP: {best_average_precision:.4f}).")
                    break
            if scheduler is not None:
                old_lrs = [group['lr'] for group in optimiser.param_groups]
                if isinstance(scheduler,torch.optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(average_precision)
                else:
                    scheduler.step()
                new_lrs = [group['lr'] for group in optimiser.param_groups]
                if old_lrs != new_lrs:
                    print(f"Learning rate changed from {old_lrs} to {new_lrs}")
    if best_model_state_dict is not None:
        model.load_state_dict(best_model_state_dict)
            
    return model,best_average_precision

In [19]:
def four_fold_cv_train(params,class_samples,num_epochs):
    lr = params["lr"]
    weight_decay = params["weight_decay"]
    batch_size = params["batch_size"] 
    
    
    roc_auc_scores = []
    average_precision_scores = []
    for train_index, val_index in skf.split(train_df,train_df["pCR"]):
        model = Model()
        model.to(device)
        optimiser = torch.optim.Adam(model.parameters(),lr=lr,weight_decay=weight_decay)
        fold_train_df = train_df.iloc[train_index]
        fold_val_df = train_df.iloc[val_index]
        fold_train_dataset = ModelDataset(fold_train_df,class_samples=class_samples,loading_bar=False,caching=True)
        fold_train_loader = DataLoader(fold_train_dataset,batch_size=batch_size,shuffle=True)
        fold_val_dataset = ModelDataset(fold_val_df,class_samples={0:1,1:1},loading_bar=False,caching=True)
        fold_val_loader = DataLoader(fold_val_dataset,batch_size=batch_size)
        model,score = trainModel(model,fold_train_loader,optimiser,device=device,num_epochs=num_epochs,val_loader=fold_val_loader,patience=num_epochs//4)
        average_precision_scores.append(score)
        roc_auc_scores.append(evaluate_roc_auc(model,fold_val_loader))
    return sum(roc_auc_scores)/len(roc_auc_scores),sum(average_precision_scores)/len(average_precision_scores)

## With Rotations

In [ ]:
roc_auc_score, averagePrecisionScore = four_fold_cv_train(best_params,class_samples={0:3,1:8},num_epochs=20)
print(roc_auc_score, averagePrecisionScore)
# 0.5754968159859464 0.4079327554507933

Dataset initialised with 409 entries.
Dataset initialised with 31 entries.
Epoch 0 Done. Avg Loss: 12.1201
Average Precision: 0.2137028440907751
Epoch 1 Done. Avg Loss: 1.2741
Average Precision: 0.26666666666666666
Epoch 2 Done. Avg Loss: 4.8314
Average Precision: 0.34846849875774555
Epoch 3 Done. Avg Loss: 5.3274
Average Precision: 0.27585057747017827
Epoch 4 Done. Avg Loss: 3.2612
Average Precision: 0.23061206504411372
Epoch 5 Done. Avg Loss: 1.3999
Average Precision: 0.26666666666666666
Epoch 6 Done. Avg Loss: 0.6673
Average Precision: 0.24454022988505747
Epoch 7 Done. Avg Loss: 7.2478
Average Precision: 0.414080459770115
Epoch 8 Done. Avg Loss: 2.4356
Average Precision: 0.27586206896551724
Epoch 9 Done. Avg Loss: 0.8802
Average Precision: 0.25806451612903225
Epoch 10 Done. Avg Loss: 0.9533
Average Precision: 0.26666666666666666
Epoch 11 Done. Avg Loss: 1.3634
Average Precision: 0.26666666666666666
Epoch 12 Done. Avg Loss: 2.0559
Average Precision: 0.24454022988505747
Early stopping

## Without Rotations

In [ ]:
roc_auc_score, averagePrecisionScore = four_fold_cv_train(best_params,class_samples={0:1,1:1},num_epochs=20)
print(roc_auc_score, averagePrecisionScore)
# 0.6127442907334211 0.48913438709321194

Dataset initialised with 93 entries.
Dataset initialised with 31 entries.
Epoch 0 Done. Avg Loss: 48.9052
Average Precision: 0.285978835978836
Epoch 1 Done. Avg Loss: 4.1351
Average Precision: 0.3283531746031746
Epoch 2 Done. Avg Loss: 2.1912
Average Precision: 0.238433229352347
Epoch 3 Done. Avg Loss: 2.6534
Average Precision: 0.22214746816825373
Epoch 4 Done. Avg Loss: 2.1207
Average Precision: 0.295280784030784
Epoch 5 Done. Avg Loss: 3.6896
Average Precision: 0.4259209631295463
Epoch 6 Done. Avg Loss: 2.5501
Average Precision: 0.29805466524216523
Epoch 7 Done. Avg Loss: 2.2496
Average Precision: 0.3125609524746354
Epoch 8 Done. Avg Loss: 0.9570
Average Precision: 0.43583638583638584
Epoch 9 Done. Avg Loss: 0.6969
Average Precision: 0.3876979501979502
Epoch 10 Done. Avg Loss: 0.9016
Average Precision: 0.373001845863688
Epoch 11 Done. Avg Loss: 1.0040
Average Precision: 0.37130158678333547
Epoch 12 Done. Avg Loss: 2.0223
Average Precision: 0.3680555555555555
Epoch 13 Done. Avg Loss: 